In [4]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Input,Dropout
from tensorflow.keras.layers import LSTM
from tensorflow.keras.models import load_model

In [2]:
## Load IMDB dataset and index
word_index = imdb.get_word_index()


reverse_word_index = {value: key for key, value in word_index.items()}

In [6]:
# load the saved model
model=load_model('imdb_lstm_model.h5')
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 256, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,415,747 (5.40 MB)

 Trainable params: 1,415,745 (5.40 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [7]:
model.get_weights()

[array([[ 0.44798082,  0.30324474, -0.35579225, ..., -0.24440096,
          0.4199318 , -0.48680186],
        [ 0.0241974 , -0.04830825,  0.00505865, ..., -0.0267506 ,
         -0.02352839,  0.0476286 ],
        [-0.04331661,  0.08095089,  0.01443743, ...,  0.03528177,
         -0.0707273 , -0.00172367],
        ...,
        [ 0.03517872,  0.00858131, -0.00298934, ...,  0.00676022,
          0.03051862, -0.01335108],
        [-0.02299116,  0.06217791,  0.0633944 , ..., -0.03685204,
         -0.04050655,  0.01686526],
        [-0.05339028, -0.08605143,  0.07814141, ...,  0.04799059,
         -0.00157834,  0.09211461]], shape=(10000, 128), dtype=float32),
 array([[ 0.06424486, -0.01068503, -0.07765498, ..., -0.05896668,
         -0.08890319,  0.01379828],
        [ 0.02455016,  0.07213621,  0.05274301, ...,  0.03351935,
          0.03647946, -0.01111091],
        [-0.16809924, -0.10885222, -0.08423001, ...,  0.09275261,
          0.12268074,  0.02593604],
        ...,
        [-0.1479735

In [21]:
import re

def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

def preprocess_review(text):
    # Strip punctuation first — "fantastic!" → "fantastic"
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    words = text.split()

    encoded_review = []
    for word in words:
        idx = word_index.get(word, 2)      # 2 = unknown
        encoded_review.append(min(idx + 3, 9999))  # +3 offset, cap at 9999

    padded_review = sequence.pad_sequences(
        [encoded_review], maxlen=256, padding='post', truncating='post'
    )
    return padded_review


In [22]:
# Prediction function
def predict_sentiment(review):
    preprocessed = preprocess_review(review)
    score = model.predict(preprocessed, verbose=0)[0][0]
    sentiment = "Positive" if score > 0.5 else "Negative"
    return sentiment, float(score)

In [27]:
# User input and prediction
user_review = input("Enter your movie review: ")

sentiment, score = predict_sentiment(user_review)

print(f"\nReview: {user_review}")
print(f"Predicted Sentiment: {sentiment}")
print(f"Confidence Score: {score:.4f}")


Review: movie is amazing, you can watch
Predicted Sentiment: Positive
Confidence Score: 0.9300
